In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

from IPython.display import HTML, display
from PIL import Image, ImageDraw, ImageFont

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent

ASSESSMENT_PATH = ROOT / 'data/camera_capture_test_scene_refactor/scene_assessment.json'
with ASSESSMENT_PATH.open(encoding='utf-8') as file:
    assessment = json.load(file)

records = [dict(image_path=image_path, **record) for image_path, record in assessment.items()]
DEFECT_CLASSES = ['erosion', 'vegetation', 'seepage', 'structure_damage', 'settlement']
STATUSES = ['ok', 'maybe', 'poor']

len(records), ASSESSMENT_PATH


In [ ]:
print('images', len(records))

for defect_class in DEFECT_CLASSES:
    counts = Counter(record['synthesis_suitability'][defect_class]['status'] for record in records)
    print(defect_class, dict(counts))


In [ ]:
top_label_counts = Counter()
for record in records:
    for item in record['top_labels']:
        top_label_counts[item['label']] += 1

for label, count in sorted(top_label_counts.items()):
    print(f'{label}	{count}')


In [ ]:
def median(values: list[float]) -> float:
    values = sorted(values)
    mid = len(values) // 2
    if len(values) % 2:
        return values[mid]
    return (values[mid - 1] + values[mid]) / 2


metric_names = [
    'water_area_ratio',
    'soft_land_area_ratio',
    'hard_surface_area_ratio',
    'road_path_area_ratio',
    'structure_area_ratio',
    'vegetation_occluder_area_ratio',
    'usable_surface_area_ratio',
    'usable_surface_largest_component_ratio',
    'water_land_boundary_ratio',
]

for name in metric_names:
    values = [record['metrics'][name] for record in records]
    print(f'{name}	min={min(values):.3f}	median={median(values):.3f}	max={max(values):.3f}')


In [ ]:
def resolve_path(path_text: str) -> Path:
    path = Path(path_text)
    if path.exists():
        return path
    candidates = [
        ROOT / 'data/camera_capture_test' / path.name,
        ROOT / 'data/camera_capture_test_scene_refactor' / path.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(path_text)


def make_pair(record: dict, title: str, image_width: int = 420, gap: int = 12, title_height: int = 42) -> Image.Image:
    image = Image.open(resolve_path(record['image_path'])).convert('RGB')
    overlay = Image.open(resolve_path(record['overlay_path'])).convert('RGB')

    image.thumbnail((image_width, image_width), Image.Resampling.LANCZOS)
    overlay.thumbnail((image_width, image_width), Image.Resampling.LANCZOS)

    width = image.width + overlay.width + gap
    height = max(image.height, overlay.height) + title_height
    canvas = Image.new('RGB', (width, height), 'white')
    canvas.paste(image, (0, title_height))
    canvas.paste(overlay, (image.width + gap, title_height))

    draw = ImageDraw.Draw(canvas)
    font = ImageFont.load_default()
    draw.text((0, 4), title[:140], fill='black', font=font)
    draw.text((0, 22), 'original', fill='black', font=font)
    draw.text((image.width + gap, 22), 'overlay', fill='black', font=font)
    return canvas


def show_pairs(selected: list[dict], title: str) -> None:
    display(HTML(f'<h3>{title}</h3>'))
    if not selected:
        print('no examples')
        return
    for record in selected:
        display(make_pair(record, Path(record['image_path']).name))


In [ ]:
EXAMPLES_PER_STATUS = 5

In [ ]:
def candidates_for(defect_class: str, status: str, limit: int = EXAMPLES_PER_STATUS) -> list[dict]:
    selected = [
        record
        for record in records
        if record['synthesis_suitability'][defect_class]['status'] == status
    ]
    return sorted(
        selected,
        key=lambda record: record['synthesis_suitability'][defect_class]['score'],
        reverse=(status != 'poor'),
    )[:limit]


for defect_class in DEFECT_CLASSES:
    display(HTML(f'<h2>{defect_class}</h2>'))
    for status in STATUSES:
        selected = candidates_for(defect_class, status)
        print(defect_class, status, [Path(record['image_path']).name for record in selected])
        show_pairs(selected, f'{defect_class}: {status}')


In [ ]:
for defect_class in DEFECT_CLASSES:
    print('\n' + defect_class)
    ranked = sorted(
        records,
        key=lambda record: record['synthesis_suitability'][defect_class]['score'],
        reverse=True,
    )
    for record in ranked[:10]:
        decision = record['synthesis_suitability'][defect_class]
        print(
            f"{decision['status']:<5} {decision['score']:.3f} "
            f"{Path(record['image_path']).name} "
            f"reasons={';'.join(decision['reasons'])}"
        )
